# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamaKousalya/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

# Find CSV files in the repo
files = list(Path(".").rglob("*.csv"))
print(files)


[PosixPath('sample_data/mnist_train_small.csv'), PosixPath('sample_data/mnist_test.csv'), PosixPath('sample_data/california_housing_train.csv'), PosixPath('sample_data/california_housing_test.csv')]


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Staleness
path = files[0]
df = pd.read_csv(path)

print("Using:", path)
print("Columns:", df.columns.tolist())
display(df.head())


Using: sample_data/mnist_train_small.csv
Columns: ['6', '0', '0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9', '0.10', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.20', '0.21', '0.22', '0.23', '0.24', '0.25', '0.26', '0.27', '0.28', '0.29', '0.30', '0.31', '0.32', '0.33', '0.34', '0.35', '0.36', '0.37', '0.38', '0.39', '0.40', '0.41', '0.42', '0.43', '0.44', '0.45', '0.46', '0.47', '0.48', '0.49', '0.50', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.60', '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.70', '0.71', '0.72', '0.73', '0.74', '0.75', '0.76', '0.77', '0.78', '0.79', '0.80', '0.81', '0.82', '0.83', '0.84', '0.85', '0.86', '0.87', '0.88', '0.89', '0.90', '0.91', '0.92', '0.93', '0.94', '0.95', '0.96', '0.97', '0.98', '0.99', '0.100', '0.101', '0.102', '0.103', '0.104', '0.105', '0.106', '0.107', '0.108', '0.109', '0.110', '0.111', '0.112', '0.113', '0.114', '0.115', '0.116', '0.

,6,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.581,0.582,0.583,0.584,0.585,0.586,0.587,0.588,0.589,0.590
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Automatically find likely columns
cols = {c.lower(): c for c in df.columns}

def find(names):
    for name in names:
        for c in df.columns:
            if name in c.lower():
                return c
    return None

position = find(["position", "rank"])
ctr = find(["ctr", "click"])
volume = find(["volume", "impression"])
fresh = find(["fresh", "age", "updated"])

print("position:", position)
print("ctr:", ctr)
print("volume:", volume)
print("freshness:", fresh)

# Signals
df["stale_flag"] = False
df["ctr_flag"] = False
df["volume_flag"] = False

if fresh:
    x = pd.to_numeric(df[fresh], errors="coerce")
    df["stale_flag"] = x > 180

if position and ctr:
    p = pd.to_numeric(df[position], errors="coerce")
    c = pd.to_numeric(df[ctr], errors="coerce")
    expected = c.groupby(p).transform("median")
    df["ctr_flag"] = c < expected * 0.75

if position and volume:
    p = pd.to_numeric(df[position], errors="coerce")
    v = pd.to_numeric(df[volume], errors="coerce")
    df["volume_flag"] = (v >= v.quantile(.75)) & p.between(4, 20)

print("\nSignal results:")
print("Stale:", df["stale_flag"].sum())
print("CTR:", df["ctr_flag"].sum())
print("Volume:", df["volume_flag"].sum())

position: None
ctr: None
volume: None
freshness: None

Signal results:
Stale: 0
CTR: 0
Volume: 0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["score"] = (
    df["stale_flag"].astype(int) * 3 +
    df["ctr_flag"].astype(int) * 3 +
    df["volume_flag"].astype(int) * 4
)

df["reason_code"] = np.select(
    [df["volume_flag"], df["ctr_flag"], df["stale_flag"]],
    ["HIGH_VOLUME_QUICKWIN", "LOW_CTR_FOR_POSITION", "STALE_REFRESH"],
    default="NONE"
)

df["action_label"] = np.where(
    df["score"] >= 6, "OPTIMIZE",
    np.where(df["score"] >= 3, "REVIEW", "MONITOR")
)

queue = df.sort_values("score", ascending=False)

display(queue.head(10))

Path("../outputs").mkdir(parents=True, exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)

print("Saved: ../outputs/baseline_action_score.csv")

,6,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.587,0.588,0.589,0.590,stale_flag,ctr_flag,volume_flag,score,reason_code,action_label
19998,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
2,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
4,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
6,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
7,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR
8,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,False,0,NONE,MONITOR


Saved: ../outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.